In [0]:
# Databricks 클러스터에 사전 설치 필요
#%pip install geopandas pyrosm


In [0]:
# Azure Storage 연결을 위한 SAS 토큰 설정
# 보안을 위해 실제 값 삭제
STORAGE_ACCOUNT = ""
CONTAINER       = ""
SAS_TOKEN       = ""

spark.conf.set(f"fs.azure.sas.{CONTAINER}.{STORAGE_ACCOUNT}.blob.core.windows.net", SAS_TOKEN)


In [0]:
# Databricks 경로
MOUNT_POINT = f"/mnt/{CONTAINER}"
# Azure Blob 스토리지 경로
SOURCE = f"wasbs://{CONTAINER}@{STORAGE_ACCOUNT}.blob.core.windows.net"
# SAS토큰으로 인증
CONF_KEY = f"fs.azure.sas.{CONTAINER}.{STORAGE_ACCOUNT}.blob.core.windows.net"

# 마운트 됐는지 확인
if not any(mount.mountPoint == MOUNT_POINT for mount in dbutils.fs.mounts()):
    dbutils.fs.mount(
        source=SOURCE,
        mount_point=MOUNT_POINT,
        extra_configs={CONF_KEY: SAS_TOKEN},
    )
    print(f"마운트 완료: {MOUNT_POINT}")
else:
    print(f"이미 마운트됨: {MOUNT_POINT}")

이미 마운트됨: /mnt/raw


In [0]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import box
from pyrosm import OSM

# 한국 전체 pbf에서 bbox로 파싱 범위 제한 — 메모리 절약
SE_SEOUL_BBOX = {
    "min_lon": 127.01, "min_lat": 37.42,
    "max_lon": 127.19, "max_lat": 37.58,
}
bbox_poly = box(
    SE_SEOUL_BBOX["min_lon"], SE_SEOUL_BBOX["min_lat"],
    SE_SEOUL_BBOX["max_lon"], SE_SEOUL_BBOX["max_lat"],
)

PBF_PATH      = "/dbfs/mnt/raw/osm/south-korea-latest.osm.pbf"
BLOB_RAW_PATH = "/dbfs/mnt/raw/osm"
CSV_BASE_PATH = "/dbfs/mnt/raw/parks"

osm = OSM(PBF_PATH, bounding_box=[
    SE_SEOUL_BBOX["min_lon"], SE_SEOUL_BBOX["min_lat"],
    SE_SEOUL_BBOX["max_lon"], SE_SEOUL_BBOX["max_lat"],
])

nodes, edges = osm.get_network(network_type="walking", nodes=True)

EDGE_COLUMNS = ["id", "u", "v", "geometry", "length", "surface", "smoothness", "highway"]
save_cols = [c for c in EDGE_COLUMNS if c in edges.columns]

edges[save_cols].to_file(f"{BLOB_RAW_PATH}/edges_final.geojson", driver="GeoJSON")
print(f"edges_final.geojson 저장 완료 ({len(edges):,}건)")

edges_for_spark = edges[save_cols].copy()

# Spark는 geometry 타입 직접 못 읽어서 WKT 문자열로 변환
edges_for_spark["geometry"] = edges_for_spark["geometry"].apply(
    lambda x: x.wkt if x is not None else None
)

edges_for_spark = edges_for_spark.where(pd.notnull(edges_for_spark), None)

spark_edges_df = spark.createDataFrame(edges_for_spark)

spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")

spark_edges_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("bronze.osm_edges")

print("✅ bronze.osm_edges 적재 완료")

edges_final.geojson 저장 완료 (224,690건)


/root/.ipykernel/1500/command-7035996664157080-2229078897:35: UserWarning: Geometry column does not contain geometry.
  edges_for_spark["geometry"] = edges_for_spark["geometry"].apply(


✅ bronze.osm_edges 적재 완료


In [0]:
from pyspark.sql.functions import col, count, when

bronze_df = spark.table("bronze.osm_edges")
print(f"총 레코드 수: {bronze_df.count():,}행")

# 칼럼별 결측치 확인
null_counts = bronze_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in bronze_df.columns
])

print("\n🔍 결측치 현황")
row = null_counts.first()
for c in bronze_df.columns:
    print(f"  {c}: {row[c]:,}건")


총 레코드 수: 224,690행

🔍 결측치 현황
  id: 0건
  u: 0건
  v: 0건
  geometry: 0건
  length: 0건
  surface: 190,923건
  smoothness: 223,323건
  highway: 0건


In [0]:
leisures = osm.get_pois(custom_filter={"leisure": True})

if leisures is None or leisures.empty:
    leisures = gpd.GeoDataFrame(columns=["name", "leisure", "geometry"], crs="EPSG:4326")
else:
    keep_cols = [c for c in ["name", "leisure", "geometry"] if c in leisures.columns]
    leisures = leisures[keep_cols].copy()

print(f"OSM leisure 추출: {len(leisures):,}건")

# OSM 공원 데이터는 이름 없거나 누락이 많아서 구청 공식 CSV로 대체
def load_official_parks(gu_name: str) -> pd.DataFrame:
    file_path = f"{CSV_BASE_PATH}/{gu_name}.csv"
    try:
        df = pd.read_csv(file_path, encoding="utf-8-sig")
    except UnicodeDecodeError:
        df = pd.read_csv(file_path, encoding="cp949")
    df.columns = df.columns.str.strip()
    df = df.rename(columns={"공원명": "name", "위도": "lat", "경도": "lon", "반려동물": "pet"})
    return df[["name", "lat", "lon", "pet"]]


park_df = pd.concat(
    [load_official_parks(gu) for gu in ["gangnam", "gangdong", "songpa"]],
    ignore_index=True
).dropna(subset=["lat", "lon"])

park_gdf = gpd.GeoDataFrame(
    park_df,
    geometry=gpd.points_from_xy(park_df["lon"], park_df["lat"]),
    crs="EPSG:4326"
)
park_gdf["leisure"] = "park"

park_gdf_clean = park_gdf.drop(columns=["lat", "lon"])
print(f"구청 공원 데이터: {len(park_gdf_clean):,}건")

combined = park_gdf_clean.copy()

# geometry → WKT 변환
combined["geometry"] = combined["geometry"].apply(
    lambda x: x.wkt if x is not None else None)

try:
    spark.createDataFrame(combined.fillna(pd.NA)) \
        .write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("bronze.osm_leisure")
    print(f"✅ bronze.osm_leisure 적재 완료 ({len(combined):,}건)")
except Exception as e:
    print(f"❌ 저장 실패: {e}")

OSM leisure 추출: 3,291건
구청 공원 데이터: 360건


/root/.ipykernel/1500/command-7035996664157083-409561742:43: UserWarning: Geometry column does not contain geometry.
  combined["geometry"] = combined["geometry"].apply(


✅ bronze.osm_leisure 적재 완료 (360건)


In [0]:
import geopandas as gpd
import pandas as pd

DBFS_RAW_DIR = "/dbfs/mnt/raw"


def save_shp_to_bronze(sub_path: str, table_name: str) -> None:
    full_path = f"{DBFS_RAW_DIR}/{sub_path}"
    print(f"[{table_name}] 로딩: {full_path}")

    gdf = gpd.read_file(full_path)
    print(f"[{table_name}] 원본 CRS: {gdf.crs}")

    # vworld SHP는 CRS 정보 누락된 파일이 있어서 없으면 5186 강제 지정
    if gdf.crs is None:
        print(f"[{table_name}] ⚠️  CRS 없음 → EPSG:5186 강제 지정")
        gdf = gdf.set_crs("EPSG:5186")

    pdf = pd.DataFrame(gdf)
    if "geometry" in pdf.columns:
        # vworld 원본은 EPSG:5181/5186 기준 → WGS84로 변환 후 WKT로 저장
        gdf_4326 = gdf.to_crs("EPSG:4326")
        pdf["geometry"] = gdf_4326["geometry"].apply(
            lambda x: x.buffer(0).wkt if x else None
        )
        pdf["crs_info"] = "EPSG:4326"

    spark.createDataFrame(pdf.where(pd.notnull(pdf), None)) \
        .write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .option("mergeSchema", "true") \
        .saveAsTable(f"bronze.{table_name}")

    print(f"[{table_name}] ✅ 완료\n")


targets = [
    ("sig/sig.shp",                                               "sig"),
    ("vworld/slope/gangnam_songpa_gangdong_soilslope.shp",         "vworld_slope"),
    ("vworld/soil_type/gangnam_songpa_gangdong_soil_type.shp",     "vworld_soil_type"),
    ("vworld/gravel/gangnam_songpa_gangdong_gravel.shp",           "vworld_gravel"),
    ("vworld/soil_depth/gangnam_songpa_gangdong_soil_depth.shp",   "vworld_soil_depth"),
    ("vworld/drainage/gangnam_songpa_gangdong_drainage_class.shp", "vworld_drainage"),
]

for sub_path, table_name in targets:
    save_shp_to_bronze(sub_path, table_name)

[sig] 로딩: /dbfs/mnt/raw/sig/sig.shp
[sig] 원본 CRS: None
[sig] ⚠️  CRS 없음 → EPSG:5186 강제 지정
[sig] ✅ 완료

[vworld_slope] 로딩: /dbfs/mnt/raw/vworld/slope/gangnam_songpa_gangdong_soilslope.shp
[vworld_slope] 원본 CRS: EPSG:5181
[vworld_slope] ✅ 완료

[vworld_soil_type] 로딩: /dbfs/mnt/raw/vworld/soil_type/gangnam_songpa_gangdong_soil_type.shp
[vworld_soil_type] 원본 CRS: EPSG:5181
[vworld_soil_type] ✅ 완료

[vworld_gravel] 로딩: /dbfs/mnt/raw/vworld/gravel/gangnam_songpa_gangdong_gravel.shp
[vworld_gravel] 원본 CRS: EPSG:5181
[vworld_gravel] ✅ 완료

[vworld_soil_depth] 로딩: /dbfs/mnt/raw/vworld/soil_depth/gangnam_songpa_gangdong_soil_depth.shp
[vworld_soil_depth] 원본 CRS: EPSG:5181
[vworld_soil_depth] ✅ 완료

[vworld_drainage] 로딩: /dbfs/mnt/raw/vworld/drainage/gangnam_songpa_gangdong_drainage_class.shp
[vworld_drainage] 원본 CRS: EPSG:5181
[vworld_drainage] ✅ 완료



In [0]:
from pyspark.sql.functions import when, col

# VLDSOILDEP 컬럼에 CP949→UTF-8 변환 오류로 '±âÅ¸'가 들어온 케이스 수정
df_fixed = spark.table("bronze.vworld_soil_depth").withColumn(
    "VLDSOILDEP",
    when(col("VLDSOILDEP") == "±âÅ¸", "기타").otherwise(col("VLDSOILDEP"))
)

df_fixed.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("bronze.vworld_soil_depth")

print("✅ VLDSOILDEP 인코딩 수정 완료")


✅ VLDSOILDEP 인코딩 수정 완료
